# 📘 Notebook 4 - Solução do Problema e Autoavaliação


---

## 💻 4.1. Consultas à camada Gold para obter as respostas

Faremos agora consultas SQL ou PySpark para obter as respostas a cada uma das nossas perguntas iniciais.

### a) Quantos pacientes foram admitidos os readmitidos ao longo do tempo?

In [0]:
%sql
SELECT
  DATE_FORMAT(Inicio, 'yyyy-MM') AS mes,
  COUNT(*) AS total_admissoes
FROM gold.flat
WHERE Inicio IS NOT NULL
GROUP BY DATE_FORMAT(Inicio, 'yyyy-MM')
ORDER BY mes;


mes,total_admissoes
2011-01,86
2011-02,86
2011-03,129
2011-04,113
2011-05,131
2011-06,131
2011-07,124
2011-08,115
2011-09,124
2011-10,84


Databricks visualization. Run in Databricks to view.

![](https://raw.githubusercontent.com/cristianofanchin/puc-rio/main/engenhariadados/plots/plot_a1.png)


In [0]:
Observa-se um possível outlier em fevereiro/2014 e março/2014. Vamos retirar esses dois meses da análise e inverter os eixos do gráfico:

In [0]:
%sql
SELECT
  DATE_FORMAT(Inicio, 'yyyy-MM') AS mes,
  COUNT(*) AS total_admissoes
FROM gold.flat
WHERE Inicio IS NOT NULL
AND DATE_FORMAT(Inicio, 'yyyy-MM') NOT IN ('2014-02', '2014-03')
GROUP BY DATE_FORMAT(Inicio, 'yyyy-MM')
ORDER BY mes;

mes,total_admissoes
2011-01,86
2011-02,86
2011-03,129
2011-04,113
2011-05,131
2011-06,131
2011-07,124
2011-08,115
2011-09,124
2011-10,84


Databricks visualization. Run in Databricks to view.

![](https://raw.githubusercontent.com/cristianofanchin/puc-rio/main/engenhariadados/plots/plot_a2.png)

Claramente, destaca-se no gráfico o período da pandemia de COVID-19 nos anos de 2020 e 2021.

### b) Qual é o tempo médio de permanência dos pacientes no hospital?

In [0]:
%sql
SELECT
  ROUND(AVG(Duracao_minutos) / 60.0, 1) AS tempo_medio_horas
FROM
  gold.flat
WHERE
  Duracao_minutos IS NOT NULL;



tempo_medio_horas
7.3


Percebemos o tempo médio de **7,3 horas**.

### c) Qual é o custo médio por visita?

In [0]:
%sql
SELECT
  ROUND(AVG(CAST(Total_Cost AS DOUBLE)), 2) AS custo_medio_por_visita
FROM
  gold.flat
WHERE
  Total_Cost IS NOT NULL;


custo_medio_por_visita
3639.68


O custo médio por visita é **US$ 3.639,68**.

### d) Qual o percentual de visitas de homens ou de mulheres?

In [0]:
%sql
SELECT
  GENDER,
  COUNT(*) AS total_visitas,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS genero_percentual
FROM
  gold.flat
WHERE
  GENDER IS NOT NULL
GROUP BY
  GENDER;


GENDER,total_visitas,genero_percentual
F,14924,53.51
M,12967,46.49


Databricks visualization. Run in Databricks to view.

%md
![](https://raw.githubusercontent.com/cristianofanchin/puc-rio/main/engenhariadados/plots/plot_d.png)

As mulheres foram a maioria.

### e) Quais os maiores motivos de visitas hospitalares?

In [0]:
%sql
SELECT 
  ReasonDescription,
  COUNT(*) AS total_visitas
FROM gold.flat
WHERE ReasonDescription IS NOT NULL AND ReasonDescription != ''
GROUP BY ReasonDescription
ORDER BY total_visitas DESC
LIMIT 10;


ReasonDescription,total_visitas
Chronic congestive heart failure (disorder),1738
Hyperlipidemia,1565
Normal pregnancy,1341
Viral sinusitis (disorder),732
Malignant neoplasm of breast (disorder),723
Acute viral pharyngitis (disorder),400
Acute bronchitis (disorder),352
Alzheimer's disease (disorder),191
Sinusitis (disorder),115
Asthma,113


Os três maiores motivos são:
- Insuficiência cardíaca congestiva crônica (distúrbio)
- Hiperlipidemia
- Gravidez normal

### f) Qual é o gasto hospitalar dos casos em que não há cobertura de seguro?

Para responder a essa pergunta, iremos buscar os dados de gastos das visitas em que o pagador (Payer) é igual a "NO_INSURANCE", com o resultado sendo apresentado em classes (_bins_) para monstarmos um histograma.

In [0]:
%sql
SELECT 
  CASE 
    WHEN CAST(Total_Cost AS DOUBLE) < 100 THEN 'a) < 100'
    WHEN CAST(Total_Cost AS DOUBLE) BETWEEN 100 AND 499.99 THEN 'b) 100 - 499'
    WHEN CAST(Total_Cost AS DOUBLE) BETWEEN 500 AND 999.99 THEN 'c) 500 - 999'
    WHEN CAST(Total_Cost AS DOUBLE) BETWEEN 1000 AND 4999.99 THEN 'd) 1.000 - 4.999'
    WHEN CAST(Total_Cost AS DOUBLE) BETWEEN 5000 AND 9999.99 THEN 'e) 5.000 - 9.999'
    ELSE 'f) >= 10.000'
  END AS faixa_custo,
  COUNT(*) AS frequencia
FROM gold.flat
WHERE 
  (Payer_Coverage IS NULL OR Payer_Coverage = '' OR CAST(Payer_Coverage AS DOUBLE) = 0)
  AND Total_Cost IS NOT NULL AND Total_Cost != ''
GROUP BY faixa_custo
ORDER BY faixa_custo;


faixa_custo,frequencia
a) < 100,1732
b) 100 - 499,4632
c) 500 - 999,1714
d) 1.000 - 4.999,2362
e) 5.000 - 9.999,724
f) >= 10.000,2422


Databricks visualization. Run in Databricks to view.

![](https://raw.githubusercontent.com/cristianofanchin/puc-rio/main/engenhariadados/plots/plot_f.png)

Percebemos que a faixa mais frequente de custos de não-segurados é de US$ 100 a US$ 499 por visita hospitalar.

### g) Em que localidades moravam os pacientes que procuraram atendimento nos anos de 2020 e 2021, auge da pandemia por COVID-19?

Iremos novamente recorrer à biblioteca _folium_ para nos ajudar a visualizar essa resposta em um mapa.


In [0]:
pip install folium 

Python interpreter will be restarted.
  Attempting uninstall: jinja2
    Found existing installation: Jinja2 2.11.3
    Not uninstalling jinja2 at /databricks/python3/lib/python3.9/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-d17ada81-1d36-46e6-a5bf-8d427d694b2e
    Can't uninstall 'Jinja2'. No files were found to uninstall.
Python interpreter will be restarted.


In [0]:
import folium
import pandas as pd
from pyspark.sql.functions import col, year

# Ler a tabela Spark existente
df = spark.table("gold.flat")

# Filtrar registros dos anos de 2020 e 2021 com base na coluna 'Inicio'
df_filtered = df.filter(
    (year(col("Inicio")) >= 2020) & (year(col("Inicio")) <= 2021)
)

# Agrupar por LAT e LON e contar ocorrências
df_grouped = df_filtered.groupBy("LAT", "LON").count()

# Converter para Pandas para visualização com folium
df_pandas = df_grouped.toPandas()

# Converter LAT e LON para float
df_pandas['LAT'] = df_pandas['LAT'].astype(float)
df_pandas['LON'] = df_pandas['LON'].astype(float)

# Criar o mapa centralizado
mapa = folium.Map(location=[df_pandas['LAT'].mean(), df_pandas['LON'].mean()], zoom_start=5)

# Adicionar bolhas proporcionais à contagem
for _, row in df_pandas.iterrows():
    folium.CircleMarker(
        location=[row['LAT'], row['LON']],
        radius=2 + (row['count'] ** 0.5) * 1.5,
        color='crimson',
        fill=True,
        fill_color='crimson',
        fill_opacity=0.6,
        popup=f"Ocorrências: {row['count']}"
    ).add_to(mapa)

mapa


Make this Notebook Trusted to load map: File -> Trust Notebook <iframe srcdoc="<!DOCTYPE html>
<html>
<head>
 
 <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
 
 <script>
 L_NO_TOUCH = false;
 L_DISABLE_3D = false;
 </script>
 
 <style>html, body {width: 100%;height: 100%;margin: 0;padding: 0;}</style>
 <style>#map {position:absolute;top:0;bottom:0;right:0;left:0;}</style>
 <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
 <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
 <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/js/bootstrap.bundle.min.js"></script>
 <script src="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.js"></script>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/css/bootstrap.min.css"/>
 <link rel="stylesheet" href="https://netdna.bootstrapcdn.com/bootstrap/3.0.0/css/bootstrap-glyphicons.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.2.0/css/all.min.css"/>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/python-visualization/folium/folium/templates/leaflet.awesome.rotate.min.css"/>
 
 <meta name="viewport" content="width=device-width,
 initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
 <style>
 #map_ce58f9c96e83e719436220d48b0c847a {
 position: relative;
 width: 100.0%;
 height: 100.0%;
 left: 0.0%;
 top: 0.0%;
 }
 .leaflet-container { font-size: 1rem; }
 </style>
 
</head>
<body>
 
 
 <div class="folium-map" id="map_ce58f9c96e83e719436220d48b0c847a" ></div>
 
</body>
<script>
 
 
 var map_ce58f9c96e83e719436220d48b0c847a = L.map(
 "map_ce58f9c96e83e719436220d48b0c847a",
 {
 center: [42.336057762612676, -71.01813490766288],
 crs: L.CRS.EPSG3857,
 ...{
 "zoom": 5,
 "zoomControl": true,
 "preferCanvas": false,
}

 }
 );

 

 
 
 var tile_layer_5d7a63b71575b5603959c4c2c8a6d4c6 = L.tileLayer(
 "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
 {
 "minZoom": 0,
 "maxZoom": 19,
 "maxNativeZoom": 19,
 "noWrap": false,
 "attribution": "\u0026copy; \u003ca href=\"https://www.openstreetmap.org/copyright\"\u003eOpenStreetMap\u003c/a\u003e contributors",
 "subdomains": "abc",
 "detectRetina": false,
 "tms": false,
 "opacity": 1,
}

 );
 
 
 tile_layer_5d7a63b71575b5603959c4c2c8a6d4c6.addTo(map_ce58f9c96e83e719436220d48b0c847a);
 
 
 var circle_marker_c4a89345e6788d640ab662596e8a8090 = L.circleMarker(
 [42.45475038910253, -71.11381795],
 {"bubblingMouseEvents": true, "color": "crimson", "dashArray": null, "dashOffset": null, "fill": true, "fillColor": "crimson", "fillOpacity": 0.6, "fillRule": "evenodd", "lineCap": "round", "lineJoin": "round", "opacity": 1.0, "radius": 9.794228634059948, "stroke": true, "weight": 3}
 ).addTo(map_ce58f9c96e83e719436220d48b0c847a);
 
 
 var popup_f4fc2acc55db62f0b0a5924b44f02c23 = L.popup({
 "maxWidth": "100%",
});

 
 
 var html_3f07f24b65f5bec64449296d8edcd96d = $(`<div id="html_3f07f24b65f5bec64449296d8edcd96d" style="width: 100.0%; height: 100.0%;">Ocorrências: 27.0</div>`)[0];
 popup_f4fc2acc55db62f0b0a5924b44f02c23.setContent(html_3f07f24b65f5bec64449296d8edcd96d);
 
 

 circle_marker_c4a89345e6788d640ab662596e8a8090.bindPopup(popup_f4fc2acc55db62f0b0a5924b44f02c23)
 ;

 
 
 
 var circle_marker_af2d32fbe43e12556208b5bb2052701c = L.circleMarker(
 [42.35365691303956, -71.03266516],
 {"bubblingMouseEvents": true, "color": "crimson", "dashArray": null, "dashOffset": null, "fill": true, "fillColor": "crimson", "fillOpacity": 0.6, "fillRule": "evenodd", "lineCap": "round", "lineJoin": "round", "opacity": 1.0, "radius": 5.9686269665968865, "stroke": true, "weight": 3}
 ).addTo(map_ce58f9c96e83e719436220d48b0c847a);
 
 
 var popup_f5f988a78b1

![](https://raw.githubusercontent.com/cristianofanchin/puc-rio/main/engenhariadados/plots/plot_g.png)

Por meio dessa imagem do mapa, podemos compreender qual foi a área de abrangência do Hospital Geral de Massashussets nos anos de 2020 e 2021, auge da pandemia.

Novamente, percebemos localizações inválidas para os endereços residenciais. Todavia, não nos impedem de visualizar o principal do gráfico, que é a maior ou menor incidência de visitas a partir das localidades de Boston e arredores.

## 📊 4.2. Autoavaliação

📌 O desenvolmento desse MVP permitiu perceber o poder da linguagem SQL e das bibliotecas Python para o trabalho de ETL e de análise de dados a partir de uma solução em nuvem.

O ambiente Databricks Community mostrou-se bastante versátil e, mesmo com as limitações de tempo de ativação de cluster e não-persistência das tabelas, foi possível cumprir todo o objetivo do MVP.
<br/><br/>

🧪 O maior aprendizado foi na construção do pipeline de dados, tema novo para o autor. O uso das técnicas de ETL em banco de dados em nuvem provê robustez e flexibilidade para o Engenheiro de Dados, que pode contar com diversas soluções comerciais para implantação do pipeline.

Foi também possível compreender com mais profundidade a modelagem em esquema estrela, que para esse dataset foi perfeitamente viável. 

A escolha do modelo _flat_ para a camada gold foi algo que gerou dúvidas no início, por imaginar que o esquema estrela precisaria ser mantido até o fim. Porém, por meio de busca em literatura, identificou-se as vantagens dessa abordagem:

- Preparar os dados para consumo analítico e visualizações rápidas;
- Evitar joins complexos no momento da análise;
- Permitir acesso rápido por ferramentas como Power BI, Tableau, Pandas, etc.
- Pode ser enriquecida com colunas calculadas e métricas agregadas.

O descarte da tabela _Procedures_ existente no dataset também permitiu construir o MVP de forma simples, sem precisar uma segunda tabela fato. Os dados contidos nessa tabela não fizeram falta para as análises que se pretendia fazer.
<br/><br/>

👀 Quanto às dificuldades, uma das mais relevantes foi o entendimento da referenciação de arquivos entre `dbfs:/tmp...` e `/dbfs/tmp...`, pois mesmo parecendo ser a mesma coisa, viu-se que há diferença. Enquanto o primeiro é mantido no desligamento do cluster, o segundo refere-se a um local do servidor ora em uso e não permanece no desligamento. O fato da cópia dos arquivos CSV estarem passando primeiro por um e depois pelo outro contribuiu para a confusão.

🤔 A necessidade de às vezes precisar rodar sucessivamente o código do notebook 3 trouxe também alguns desafios, pois mesmo fazendo `DROP DATABASE` das bases, na hora da recriação ocorria um erro para a camada Silver. Em uma primeira abordagem, solucionou-se isso apagando os arquivos contidos em `/user/hive/warehouse/`. Posteriormnete a solução adotada foi executar `CREATE DATABASE`, logo em seguida o `DROP DATABASE` - com isso o código original passava a funcionar.
<br/><br/>

💡 Como trabalhos futuros, pode se pensar em criar um notebook que promova cargas incrementais nas camadas Bronze, Silver e Gold a partir de novos daodos que se tornem disponíveis.

Da mesma forma, é interessante testar o processo com bases de dados maiores, para verificação da performance e dos tempos de execução.

## ✅ Fim!
Você chegou ao final desse MVP.
Espero que tenha gostado!

<br>


> [◀](https://github.com/cristianofanchin/puc-rio/blob/main/engenhariadados/README.md) Volte para o início 
